In [23]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import mlflow
import mlflow.sklearn

In [24]:
df = pd.read_csv("../data/raw/House_Prices.csv")
df = df.iloc[:1460].copy()

In [25]:
df["TotalArea"] = df[["TotalBsmtSF", "1stFlrSF", "2ndFlrSF"]].sum(axis=1, skipna=True)

df["TotalBathrooms"] = (
    df["FullBath"].fillna(0)
    + 0.5 * df["HalfBath"].fillna(0)
    + df["BsmtFullBath"].fillna(0)
    + 0.5 * df["BsmtHalfBath"].fillna(0)
)

df["HouseAge"] = df["YrSold"] - df["YearBuilt"]

df["YearsSinceRemodel"] = df["YrSold"] - df["YearRemodAdd"]

df["TotalPorchArea"] = (
    df["WoodDeckSF"]
    + df["OpenPorchSF"]
    + df["EnclosedPorch"]
    + df["3SsnPorch"]
    + df["ScreenPorch"]
)

df.loc[df["HouseAge"] < 0, "HouseAge"] = np.nan
df.loc[df["YearsSinceRemodel"] < 0, "YearsSinceRemodel"] = np.nan

In [26]:
# remove outliers that has higher GrLivArea with less saleprice
df = df.drop(index=[1298, 523])

df = df.drop(columns="Id")
df["MSSubClass"] = df["MSSubClass"].astype("str")

# fill missing values in categorical columns with none
none_cols = [
    "PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
    "GarageQual", "GarageCond", "GarageFinish", "GarageType",
    "BsmtCond", "BsmtQual", "BsmtFinType1"
]
df[none_cols] = df[none_cols].fillna("None")

# fill missing values in num columns with 0
zero_cols = [
    "BsmtFullBath", "BsmtHalfBath", "BsmtFinSF1",
    "GarageArea", "GarageCars"
]
df[zero_cols] = df[zero_cols].fillna(0)

# if the garage built year greater than the sold year mark the values as missing
df.loc[df["GarageYrBlt"] > df["YrSold"], "GarageYrBlt"] = np.nan


In [27]:
X = df.drop(columns="SalePrice")
y = df["SalePrice"]

In [28]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [29]:
numeric_cols = X_train.select_dtypes(include="number").columns
categorical_cols = X_train.select_dtypes(exclude="number").columns

In [30]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

In [31]:
gradient_boosting_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingRegressor(random_state=42))
])

In [32]:
mlflow.set_tracking_uri(
    "sqlite:////home/elgmouri/projects/aqari/notebooks/mlflow.db"
)
mlflow.set_experiment("Aqari Model Comparison")
with mlflow.start_run(run_name="Gradient Boosting"):
    gradient_boosting_pipeline.fit(X_train, y_train)
    y_pred = gradient_boosting_pipeline.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    mlflow.log_params(
        gradient_boosting_pipeline.named_steps["model"].get_params()
    )

    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("RMSE", rmse)
    mlflow.log_metric("R2", r2)
    mlflow.sklearn.log_model(
        gradient_boosting_pipeline,
        name="model",
        serialization_format="cloudpickle"
    )

2026/09/25 15:21:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


In [33]:
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

MAE: 14531.808750105632
RMSE: 20072.89440595186
R²: 0.9270563038889157
